# Pre-Entry Price Action Phase 1
事前登録Plan: `41a45c04d51f5be9bdc8c08aeb4c8cc398dcd787`。
固定28戦略・16,298 trades。Baseline再計算なし。結果・定義・coverage・正式判定は本文に表示。
PDRP60 / PDRP180: Entry直前60/180分のcompleted M1のみ。Long=(Cpre-L)/(H-L)、Short=(H-Cpre)/(H-L)。
48/144本以上、最後のM1開始時刻はEntryの1〜5分前。区分: [0,1/3), [1/3,2/3), [2/3,1]。
2022–2026は既閲覧。Phase 1は診断のみ、Volatilityとの交差分析・Entry/Risk変更なし。
CSV出力 `/content`。Driveへの保存は初期OFF。元データをアップロードするかDrive読取り用に接続し、下のパスを指定する。

In [ ]:
from pathlib import Path
import sys, urllib.request, json
import pandas as pd
from IPython.display import display
REPO = 'TR-KJ/time-entry-portfolio-lab'
BRANCH = 'research/pre-entry-price-action-phase1'
# Published run record pins the reviewed implementation. No baseline recalculation.
record_url=f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/results/pre_entry_phase1/pre_entry_phase1_run_record.csv'
published_record=pd.read_csv(record_url)
IMPLEMENTATION_SHA=str(published_record.loc[0,'ImplementationCommit'])
ROOT=Path('/content/pre_entry_phase1_code')
files=['src/research/pre_entry_price_action_phase1.py','src/research/pre_entry_phase1_frozen_inputs.json','src/research/trend_strength_phase1.py','src/research/trend_strength_phase1_frozen_inputs.json','tests/test_pre_entry_phase1.py','tests/verify_pre_entry_phase1.py']
for name in files:
    target=ROOT/name; target.parent.mkdir(parents=True,exist_ok=True)
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/{REPO}/{IMPLEMENTATION_SHA}/{name}',target)
sys.path[:0]=[str(ROOT/'src/research'),str(ROOT/'tests')]
from pre_entry_price_action_phase1 import run
BASELINE=Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT=Path('/content/m1')
OUTPUT_DIR=Path('/content')
RUN_ANALYSIS=False  # Set True after providing the exact hash-matching inputs.
SAVE_TO_DRIVE=False
if RUN_ANALYSIS:
    tables=run(BASELINE,M1_ROOT,OUTPUT_DIR,IMPLEMENTATION_SHA)
else:
    names=['strategy_primary','strategy_robustness','group_summary','period_summary','coverage','run_record','decision','combined_decision','verification']
    tables={name:pd.read_csv(f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/results/pre_entry_phase1/pre_entry_phase1_{name}.csv') for name in names}

## Portfolio: Primary / Robustness とstrategy equal-weighted

In [ ]:
display(tables['combined_decision']); display(tables['group_summary'].query("Group == 'Portfolio'"))

## 事前固定group（探索的）

In [ ]:
display(tables['decision'].query("Period == 'ALL'")); display(tables['group_summary'])

## 全28 strategyと個別主要所見（多重比較未調整）

In [ ]:
for method in ['primary','robustness']:
    print(method); display(tables['strategy_'+method]); print('Eligible clear contrasts'); display(tables['strategy_'+method].query('ClearContrast == True').drop_duplicates('StrategyNo'))

## Coverage / 欠損理由 / 固定期間 / 検証記録

In [ ]:
display(tables['coverage'].query("Period == 'ALL' and Scope == 'Portfolio'")); display(tables['period_summary']); display(tables['verification']); display(tables['run_record'].T)

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    destination=Path('/content/drive/MyDrive/time-entry-portfolio-lab/pre_entry_phase1')
    destination.mkdir(parents=True,exist_ok=True)
    for source in OUTPUT_DIR.glob('pre_entry_phase1_*.csv'):
        shutil.copy2(source,destination/source.name)